In [1]:
from typing import TypedDict
import requests
import os
from langgraph.graph import StateGraph, END 

In [2]:
SAIS_APP_URL = os.getenv("SAIS_APP_URL")
SAIS_TOKEN = os.getenv("SAIS_TOKEN")
SAIS_MODEL = os.getenv("SAIS_MODEL", "gpt-4.1")

if not SAIS_APP_URL or not SAIS_TOKEN:
    raise SystemExit("Error: SAIS_APP_URL and SAIS_TOKEN environment variables must be set.")

PROXY_HOST = os.getenv("PROXY_HOST")
PROXY_PORT = os.getenv("PROXY_PORT")
PROXY_ENABLED = os.getenv("PROXY_ENABLED", "false").lower() == "true"
SSL_VERIFY = os.getenv("SSL_VERIFY", "true").lower() == "true"

In [3]:
def generate_answer(context: str, query: str) -> str:
    response = requests.post(
        f"{SAIS_APP_URL}/v1/responses",
        headers={
            "Authorization": f"Bearer {SAIS_TOKEN}",
            "Content-Type": "application/json",
            "ApplicationType": "BRProduct",
        },
        json={
            "model": SAIS_MODEL,
            "instructions": "You are an AI technical support assistant.\n\nAnswer the user's question to the best of your knowledge.",
            "input": context + "\n\n" + query,
        },
        verify=SSL_VERIFY,
        proxies={
            "http": f"http://{PROXY_HOST}:{PROXY_PORT}" if PROXY_ENABLED else None,
            "https": f"http://{PROXY_HOST}:{PROXY_PORT}" if PROXY_ENABLED else None,
        }
    )
    if response.status_code == 200:
        response_data = response.json()
        try:
            outputs = response_data["body"]["output"]
            texts = []
            for item in outputs:
                if item.get("type") == "message":
                    for content in item.get("content", []):
                        if content.get("type") == "output_text":
                            texts.append(content.get("text", ""))
            if texts:
                return "\n".join(texts)
            return "Error: No output text found in the response."
        except KeyError:
            return "Error: Unexpected response format."
    else:
        return f"Error: Request failed with status code {response.status_code}"

In [ ]:
class Mystate(TypedDict):
    topic: str
    response: str

def generate_llm_outline(state:Mystate) -> Mystate:
    context = "You are an AI technical support assistant. Generate a concise outline for the topic provided."
    response = generate_answer(context, state["topic"])
    return Mystate(topic=state["topic"], response=response)